# 02 · Vectorizar el corpus

**Duración estimada:** 35 minutos

## Propósito

Aplicar al corpus real las dos herramientas del teórico: bag of words y TF-IDF.

Van a funcionar. Y después vamos a medir **por qué una de las dos casi no sirve
acá**, y qué se usa en su lugar.

## Qué consume

- `datos/corpus_limpio.jsonl`
- `datos/stopwords_es.txt`

## Celda de reanudación

In [ ]:
# pandas: para las tablas.
import pandas as pd

# numpy: para las cuentas sobre la matriz.
import numpy as np

# Counter: para contar.
from collections import Counter

corpus = pd.read_json("datos/corpus_limpio.jsonl", lines=True)

archivo = open("datos/stopwords_es.txt", encoding="utf-8")
palabras_vacias = archivo.read().split()
archivo.close()

print("párrafos:", f"{len(corpus):,}")
print("palabras vacías:", len(palabras_vacias))

## 1. Elegir la unidad de análisis

El corpus tiene 5.428 párrafos, cada uno con su año. Eso permite dos formas de
mirarlo:

- **Doce documentos**, uno por año, juntando todos los párrafos de cada año.
- **Cinco mil documentos**, uno por párrafo.

No es un detalle técnico: es **la decisión más importante del cuaderno**, y cambia
qué preguntas se pueden hacer.

Empecemos por los años, que es lo natural si la pregunta es sobre el paso del
tiempo.

In [ ]:
anios = sorted(corpus["anio"].unique())

documentos_por_anio = []
for anio in anios:
    parrafos_del_anio = corpus[corpus["anio"] == anio]
    texto_del_anio = " ".join(parrafos_del_anio["texto_limpio"])
    documentos_por_anio.append(texto_del_anio)

print("documentos:", len(documentos_por_anio))
print()
for posicion in range(len(anios)):
    palabras = len(documentos_por_anio[posicion].split())
    print(anios[posicion], "->", f"{palabras:,}", "palabras")

## 2. La matriz documento-término real

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# min_df=2 descarta las palabras que aparecen en un solo documento.
# Es una decisión: reduce el ruido y también borra lo muy específico.
contador = CountVectorizer(stop_words=palabras_vacias, min_df=2)

matriz = contador.fit_transform(documentos_por_anio)

print("forma de la matriz:", matriz.shape)
print("   12 documentos por", f"{matriz.shape[1]:,}", "palabras distintas")

En el teórico la matriz era de 4 por 8. Acá es de 12 por más de trece mil.

Acordate de lo que vimos sobre matrices esparsas: la mayoría de los casilleros
está en cero. Midámoslo.

In [ ]:
celdas_totales = matriz.shape[0] * matriz.shape[1]
celdas_con_valor = matriz.nnz

print("casilleros en total :", f"{celdas_totales:,}")
print("casilleros con valor:", f"{celdas_con_valor:,}")
print("porcentaje lleno    :", f"{100 * celdas_con_valor / celdas_totales:.1f}%")

Un 36% lleno. Guardar esta matriz como tabla normal costaría casi el triple de
memoria.

Y ese 36% es alto porque son doce documentos enormes. Con los párrafos como
unidad, la matriz va a estar mucho más vacía —lo vamos a ver en un rato.

## 3. TF-IDF sobre el corpus

Ahora la pregunta que TF-IDF promete contestar: **¿qué palabras caracterizan a
cada año?**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizador = TfidfVectorizer(stop_words=palabras_vacias, min_df=2)
pesos = vectorizador.fit_transform(documentos_por_anio)

vocabulario = vectorizador.get_feature_names_out()
tabla_pesos = pesos.toarray()

# Para cada año, las seis palabras con más peso.
filas = []
for posicion in range(len(anios)):
    pesos_del_anio = tabla_pesos[posicion]
    orden = np.argsort(pesos_del_anio)
    mejores = []
    for indice in orden[-6:]:
        mejores.append(vocabulario[indice])
    mejores.reverse()
    fila = {"anio": anios[posicion], "palabras_con_mas_peso": ", ".join(mejores)}
    filas.append(fila)

pd.DataFrame(filas)

**Miralo con desconfianza.**

`si`, `dos`, `vez`, `años`, `había` aparecen en casi todos los años. Son palabras
que no caracterizan nada: están en todo el corpus.

Se filtra algo de señal —`orsai` y `revista` en 2010 y 2011, `alex` y `lucas` en
2004— pero está tapada.

TF-IDF debería haber castigado justamente a las palabras que están en todos los
documentos. **No lo hizo.** Averigüemos por qué.

## 4. Por qué TF-IDF casi no sirve acá

Miremos los valores de IDF que calculó.

In [ ]:
idf_por_anios = np.round(vectorizador.idf_, 3)

print("términos en el vocabulario:", f"{len(idf_por_anios):,}")
print("valores de IDF distintos  :", len(set(idf_por_anios)))

**Trece mil quinientos términos, y solo once valores de IDF distintos.**

Tiene sentido si lo pensás: el IDF depende de en cuántos documentos aparece la
palabra. Con doce documentos, ese número solo puede ser 1, 2, 3… hasta 12. **Doce
posibilidades, once valores** (min_df=2 descarta el caso de 1).

Veamos cómo se reparten.

In [ ]:
reparto = Counter(idf_por_anios)

filas = []
for valor in sorted(reparto):
    fila = {"idf": valor, "cuantos_terminos": reparto[valor]}
    filas.append(fila)

pd.DataFrame(filas)

**Cinco mil términos —más de un tercio del vocabulario— comparten exactamente
el mismo IDF.** Para TF-IDF son todos igual de informativos.

Cuando el IDF no distingue, lo único que queda decidiendo es el TF: la frecuencia
cruda. Y volvemos al problema del cuaderno anterior.

> **La regla es simple: con N documentos, el IDF no puede distinguir más de N+1
> grupos.** TF-IDF necesita muchos documentos. Doce no alcanzan.

### Comprobarlo cambiando la unidad

Si la explicación es correcta, con los párrafos como documentos el IDF debería
tomar muchos más valores. Probemos.

In [ ]:
vectorizador_parrafos = TfidfVectorizer(stop_words=palabras_vacias, min_df=2)
vectorizador_parrafos.fit(corpus["texto_limpio"])

idf_por_parrafos = np.round(vectorizador_parrafos.idf_, 3)

filas = [
    {
        "unidad": "año",
        "documentos": len(documentos_por_anio),
        "terminos": len(idf_por_anios),
        "valores_de_idf": len(set(idf_por_anios)),
    },
    {
        "unidad": "párrafo",
        "documentos": len(corpus),
        "terminos": len(idf_por_parrafos),
        "valores_de_idf": len(set(idf_por_parrafos)),
    },
]

pd.DataFrame(filas)

De once valores a más de doscientos.

**La herramienta no cambió. Cambió la forma del corpus.**

Y esto no significa que haya que usar párrafos siempre: significa que la unidad de
análisis y la herramienta tienen que ir juntas. Si tu pregunta es sobre años,
TF-IDF entre años no es la herramienta.

## 5. La herramienta que sí sirve para comparar doce grupos

Se llama **keyness**, y es más simple que TF-IDF: no hace falta ninguna librería.

La idea cabe en una frase: **una palabra caracteriza a un año si aparece mucho más
en ese año que en todos los demás juntos.**

Se calcula comparando dos frecuencias relativas —las mismas del cuaderno
anterior— y dividiendo una por la otra.

In [ ]:
# Para cada año, contamos sus palabras con contenido.
conteos_por_anio = {}
totales_por_anio = {}

for anio in anios:
    parrafos_del_anio = corpus[corpus["anio"] == anio]
    palabras_del_anio = []
    for texto in parrafos_del_anio["texto_limpio"]:
        for palabra in texto.split():
            if palabra in palabras_vacias:
                continue
            if len(palabra) < 3:
                continue
            palabras_del_anio.append(palabra)
    conteos_por_anio[anio] = Counter(palabras_del_anio)
    totales_por_anio[anio] = len(palabras_del_anio)

for anio in anios:
    print(anio, "->", f"{totales_por_anio[anio]:,}", "palabras con contenido")

In [ ]:
def palabras_caracteristicas(anio_elegido, minimo_de_apariciones, cuantas):
    # Compara la frecuencia de cada palabra en un año contra la del resto.

    # Primero juntamos todos los años menos el elegido.
    conteo_del_resto = Counter()
    total_del_resto = 0
    for anio in anios:
        if anio == anio_elegido:
            continue
        conteo_del_resto.update(conteos_por_anio[anio])
        total_del_resto = total_del_resto + totales_por_anio[anio]

    filas = []
    for palabra in conteos_por_anio[anio_elegido]:
        veces_aca = conteos_por_anio[anio_elegido][palabra]
        if veces_aca < minimo_de_apariciones:
            continue
        frecuencia_aca = 10000 * veces_aca / totales_por_anio[anio_elegido]
        frecuencia_alla = 10000 * conteo_del_resto[palabra] / total_del_resto
        # El 0.5 evita dividir por cero cuando la palabra no está en el resto.
        razon = frecuencia_aca / (frecuencia_alla + 0.5)
        fila = {
            "palabra": palabra,
            "veces": veces_aca,
            "por_10k_este_anio": round(frecuencia_aca, 1),
            "por_10k_el_resto": round(frecuencia_alla, 1),
            "cuantas_veces_mas": round(razon, 1),
        }
        filas.append(fila)

    tabla = pd.DataFrame(filas)
    tabla = tabla.sort_values("cuantas_veces_mas", ascending=False)
    return tabla.head(cuantas).reset_index(drop=True)


palabras_caracteristicas(2010, minimo_de_apariciones=10, cuantas=10)

Comparalo con lo que había dado TF-IDF para 2010: `revista, chiri, si, años,
orsai, cada`.

Ahora aparecen `hornby`, `libreros`, `librerías`, `picnic`. **Es 2010 y no otro
año.**

Veamos los doce.

In [ ]:
filas = []
for anio in anios:
    tabla_del_anio = palabras_caracteristicas(anio, minimo_de_apariciones=10, cuantas=6)
    fila = {
        "anio": anio,
        "palabras_caracteristicas": ", ".join(tabla_del_anio["palabra"]),
    }
    filas.append(fila)

pd.DataFrame(filas)

Ahí hay una historia legible.

Hasta 2009 aparecen nombres de personajes: son los años de los relatos. Desde
2010 el vocabulario cambia a `libreros`, `revista`, `número`, `suscripción`,
`distribuidores`: son los años en que Casciari funda Editorial Orsai y se pone a
publicar.

**Eso no lo dijimos nosotros: lo dijeron los datos.** Pero fijate que la
interpretación sí es nuestra, y que hace falta saber quién es Casciari para
hacerla. Un número no interpreta solo.

> ### Perilla
>
> `palabras_caracteristicas` tiene dos parámetros y los dos cambian el resultado.
>
> **`minimo_de_apariciones`** filtra las palabras raras. Con `2`, una palabra que
> aparece dos veces en un año y nunca más va a encabezar el ranking, y no
> caracteriza nada: es una anécdota. Con `50`, solo sobreviven palabras muy
> frecuentes y desaparece lo específico.
>
> **`cuantas`** es cuántas mostrás. Ahí no hay trampa, pero sí una tentación:
> mostrar solo las que confirman lo que querías decir.
>
> Corré la celda con distintos mínimos y mirá cómo cambia la historia de 2013.

In [ ]:
for minimo in [3, 10, 30]:
    tabla_del_anio = palabras_caracteristicas(2013, minimo_de_apariciones=minimo, cuantas=6)
    print("mínimo =", minimo, "->", ", ".join(tabla_del_anio["palabra"]))

## Lo que hiciste hoy

- Elegiste la unidad de análisis, y viste que es una decisión y no un trámite.
- Armaste la matriz documento-término del corpus real y mediste su esparsidad.
- Corriste TF-IDF por año y el resultado fue pobre.
- Mediste por qué: **con doce documentos el IDF solo puede tomar once valores**, y
  un tercio del vocabulario comparte el mismo.
- Comprobaste la explicación cambiando la unidad a párrafos.
- Calculaste *keyness* con dos frecuencias relativas y una división, sin librería,
  y obtuviste algo que sí se puede leer.
- Viste que la herramienta correcta depende de la forma del corpus, no de cuál
  esté más de moda.

## Lo que viene

Tenemos resultados. Falta lo más difícil: **mostrarlos sin mentir.**